# LAB08 - Tokenization, attention mask và MLM

Kiểm tra tokenizer thật của BERT-Tiny. Các phép tính là tham chiếu kiểm chứng có AI hỗ trợ; sinh viên tự trình bày MLM calculation theo W8 mục 24. Không đoán tokenization từ ví dụ minh họa trong đề.

In [1]:
from pathlib import Path
import sys, json, csv
import numpy as np
import torch
ROOT=Path.cwd() if (Path.cwd()/"pretrained_classification.py").exists() else Path.cwd()/"lab8"
assert (ROOT/"model/config.json").exists()
sys.path.insert(0,str(ROOT))
from pretrained_classification import get_tokenizer, load_classifier
torch.set_num_threads(4)
tokenizer=get_tokenizer()
print("Tokenizer:",type(tokenizer).__name__,"vocab:",len(tokenizer))


Tokenizer: BertTokenizer vocab: 30522


## Exercise 2: The cats are playing.

So sánh words theo whitespace, content WordPiece tokens, continuation pieces (##) và special tokens. Không đồng nhất các cách đếm.

In [2]:
text="The cats are playing."
batch=tokenizer([text,"A good movie."],padding="max_length",max_length=12,truncation=True,return_tensors="pt")
print("keys:",list(batch.keys()))
for key,value in batch.items(): print(key,value.tolist())
tokens=tokenizer.convert_ids_to_tokens(batch["input_ids"][0].tolist())
print("tokens:",tokens)
active=[int(i) for i,m in zip(batch["input_ids"][0],batch["attention_mask"][0]) if m]
content=[i for i in active if i not in tokenizer.all_special_ids]
print("whitespace words:",len(text.split()),"content WordPiece:",len(content),"valid incl specials:",len(active),"padded length:",len(tokens))
print("continuation pieces:",sum(t.startswith("##") for t in tokens))
print("special tokens:",tokenizer.special_tokens_map)
assert batch["attention_mask"][0].tolist()==[1]*7+[0]*5
assert batch["input_ids"][0].tolist()==[101,1996,8870,2024,2652,1012,102,0,0,0,0,0]


keys: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids [[101, 1996, 8870, 2024, 2652, 1012, 102, 0, 0, 0, 0, 0], [101, 1037, 2204, 3185, 1012, 102, 0, 0, 0, 0, 0, 0]]
token_type_ids [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
attention_mask [[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]]
tokens: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
whitespace words: 4 content WordPiece: 5 valid incl specials: 7 padded length: 12
continuation pieces: 0
special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


`input_ids` là chỉ số vocabulary; `attention_mask`: 1=valid, 0=padding. `token_type_ids` đánh dấu segment cho sentence pair. [MASK] và attention mask có chức năng khác nhau. Một câu có thể không có continuation piece ## nhưng vẫn được tokenize bằng subword vocabulary.

In [3]:
pair=tokenizer("The movie is good.","I recommend it.")
print(tokenizer.convert_ids_to_tokens(pair["input_ids"]))
print("token_type_ids:",pair["token_type_ids"])
for text in ["unbelievably entertaining", "antidisestablishmentarianism", "NLP 😊"]:
    print(text,"->",tokenizer.tokenize(text))


['[CLS]', 'the', 'movie', 'is', 'good', '.', '[SEP]', 'i', 'recommend', 'it', '.', '[SEP]']
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
unbelievably entertaining -> ['un', '##bel', '##ie', '##va', '##bly', 'entertaining']
antidisestablishmentarianism -> ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']
NLP 😊 -> ['nl', '##p', '[UNK]']


## Exercise 1: kiểm chứng MLM loss

Đây là ba xác suất giả định đề cho, không phải probability BERT. Với một masked position, tổng và mean trên vị trí mask bằng nhau.

In [4]:
for p in [0.70,0.95,0.01]:
    print(f"P(correct)={p:.2f}: -ln(p)={-np.log(p):.9f}")
logits=torch.log(torch.tensor([[.10,.05,.70,.15]]))
loss=torch.nn.functional.cross_entropy(logits,torch.tensor([2]))
print("PyTorch CE target=eats:",loss.item())
assert abs(loss.item()+np.log(.70))<1e-6


P(correct)=0.70: -ln(p)=0.356674944
P(correct)=0.95: -ln(p)=0.051293294
P(correct)=0.01: -ln(p)=4.605170186
PyTorch CE target=eats: 0.3566749691963196


## MLM inference trên checkpoint pretrained thật

Chạy forward với MLM head từ model đã tải. Không fine-tune MLM và không gán probability đo được thành số giả định 0.70 của đề.

In [5]:
from transformers import AutoModelForMaskedLM
mlm,loading=AutoModelForMaskedLM.from_pretrained(str(ROOT/"model"),local_files_only=True,output_loading_info=True)
assert not loading["missing_keys"], "MLM weights must come from the pretrained checkpoint"
mlm.eval()
masked=tokenizer("The cat [MASK] fish.",return_tensors="pt")
position=(masked["input_ids"][0]==tokenizer.mask_token_id).nonzero().item()
with torch.inference_mode(): probs=mlm(**masked).logits[0,position].softmax(-1)
values,ids=probs.topk(5)
print("top5:",[(tokenizer.convert_ids_to_tokens(int(i)),round(float(v),5)) for i,v in zip(ids,values)])
print("P(eats) thực đo:",probs[tokenizer.convert_tokens_to_ids("eats")].item())
del mlm


top5: [('.', 0.04473), ('-', 0.03812), ('are', 0.03692), ('##fish', 0.02931), ('fish', 0.02078)]
P(eats) thực đo: 0.0009117163135670125


## Attention mask và invariance đối với padding

Cùng câu được infer riêng và trong batch có câu dài; valid logits phải gần bằng nhau khi eval mode và mask đúng.

In [6]:
model,config=load_classifier()
short="A good movie."
single=tokenizer([short],return_tensors="pt")
padded=tokenizer([short,"The acting is wonderful and the story is memorable."],padding=True,return_tensors="pt")
with torch.inference_mode():
    alone=model(**single)[0]
    together=model(**padded)[0]
print("max logits delta:",(alone-together).abs().max().item())
torch.testing.assert_close(alone,together,atol=1e-5,rtol=1e-5)
print("PASS padding invariance")


max logits delta: 2.384185791015625e-07
PASS padding invariance


Tự giải thích bằng lời: tokenization liên hệ embedding ra sao; vì sao cần attention mask; subword giúp gì với rare word; tại sao CLS chỉ trở thành useful representation thông qua pretraining/task training. Đọc `calculations.md` và tự hoàn thiện phần cá nhân theo đề.